## Model predictions
Import libraries

In [1]:
import os
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from silx.io.dictdump import h5todict

Define inputs

In [2]:
model_name = "meno"
# Data indices
indices = {
  "x": [list(range(4)),[4],[5],[6]],
  "y": [list(range(7,11)), [11]]
}
# Paths
model_name = "meno"
# Paths
path_to_datagen = "./../datagen/"
paths = {
  "data": path_to_datagen + "/data/test/",
  "model": "./../03.merged/case.r01/saving/pollu/SavedModel/",
  "indices": path_to_datagen + "/data/physics/species_indices.hdf5",
  "init_sols": path_to_datagen + "/datagen/samples/init_sols.hdf5",
  "saving": "./output/visual/"
}
os.makedirs(paths["saving"], exist_ok=True)
# Plotting
nb_eqs = 20
labels = {f"y{i+1}": fr"y$_{i+1}$" for i in range(nb_eqs)}
# Cases
cases = [3, 14, 15, 41, 53, 96]

In [3]:
# Data indices
species_indices = h5todict(paths["indices"])
indices = {
  "x": [
    species_indices["lin"],       # Linear species at time t=0
    species_indices["nlin"],      # Nonlinear species at time t=0
    species_indices["inp"],       # Input species at time t=0
    np.array([20])                # Time
  ],
  "y": [
    species_indices["lin"]+21,    # Linear species at time t
    species_indices["nlin"]+21    # Nonlinear species at time t
  ]
}

In [4]:
# Time grid
tlim = [1e-7, 60.0]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])
model = tf.keras.models.load_model(paths["model"])
init_sols = h5todict(paths["init_sols"])

Set variables names

In [ ]:
y0_names = np.array([f"y{i}" for i in species_indices["inp"]+1])
y0_names

Plot true vs. predicted solutions

In [7]:
y0 = []
yindices = np.concatenate(indices["y"])-21
for (icase, idata) in data.items():
  idata = idata.to_numpy()
  case_nb = int(icase.split("_")[-1])
  if ((cases is not None) and (case_nb not in cases)):
    continue
  _, y_true, y_pred = utils.predict(model, idata, indices, t)
  y0i = np.take(idata, species_indices["inp"], axis=-1)[0]
  y0.append(y0i)
  for yscale in ("linear", "log"):
    utils.plot_test(
      path=paths["saving"],
      x=t,
      y=np.take(y_true, yindices, axis=-1),
      y_pred=np.take(y_pred, yindices, axis=-1),
      figname=icase+"_"+yscale,
      figext=["pdf", "pgf"],
      labels=[r"$t$ [min]", r"$y$ [ppm]"],
      scales=["log", yscale],
      xlim=tlim,
      ylim=[1e-19, np.amax(y_true)*2e0] if (yscale == "log") else None,
      save=True,
      show=False
    )

In [8]:
y0_df = pd.DataFrame(np.vstack(y0), columns=y0_names, index=cases)
y0_df.to_csv(paths["saving"]+"/init_sols.csv", float_format="%.6e")